# 観測予測モデル（自己回帰・マスク予測・拡散）

観測予測では「どの条件で次を当てるか」の設計が重要です。自己回帰・マスク予測・拡散は同じ予測問題に対して異なる情報の使い方をします。


## 背景と目的

モデル選択を誤ると、短期予測は当たっても長期ロールアウトで崩れます。

三つの予測方式を同じ系列データで比較できると、タスクに合わせた設計判断がしやすくなります。

ここでは1次元系列を使い、各方式の誤差特性を比較します。


In [ ]:
import numpy as np
np.random.seed(1)

T = 80
t = np.arange(T)
series = np.sin(t / 6.0) + 0.1 * np.random.randn(T)


In [ ]:
# 1) 自己回帰: x_t = a*x_{t-1} + b
x = series[:-1]
y = series[1:]
a = np.dot(x, y) / np.dot(x, x)
b = y.mean() - a * x.mean()
ar_pred = a * x + b
ar_mse = np.mean((ar_pred - y) ** 2)

# 2) マスク予測: 中央点を前後平均で補完
mask_idx = np.arange(1, T - 1, 5)
masked = series.copy()
masked[mask_idx] = np.nan
filled = masked.copy()
for i in mask_idx:
    filled[i] = 0.5 * (masked[i - 1] + masked[i + 1])
mask_mse = np.mean((filled[mask_idx] - series[mask_idx]) ** 2)

# 3) 拡散風予測: ノイズ付加 -> 逐次平滑で復元
noisy = series + 0.35 * np.random.randn(T)
den = noisy.copy()
for _ in range(20):
    den[1:-1] = 0.25 * den[:-2] + 0.5 * den[1:-1] + 0.25 * den[2:]
diff_mse = np.mean((den - series) ** 2)

print('AR MSE         =', round(ar_mse, 6))
print('Mask-fill MSE  =', round(mask_mse, 6))
print('Diffusion-like =', round(diff_mse, 6))


自己回帰は逐次予測に強く、マスク予測は欠損補完に強い設計です。拡散型はノイズ耐性が高い一方で反復計算コストが増えます。
